In [7]:
import pandas as pd
import numpy as np
from experiment_runner import LLMExperimentRunner
from experiment_prompts import get_gamblers_fallacy_experiments, get_loss_aversion_experiments, get_serial_position_effect_experiments, get_big_five_aspects_questionaire
from system_prompts import get_persona_system_prompt
import random
import json
from concurrent.futures import ThreadPoolExecutor

In [2]:
def create_word_list(n_words):

    WORD_POOL = [
        "apple", "river", "chair", "mountain", "window",
        "pencil", "forest", "bottle", "garden", "clock",
        "bridge", "orange", "blanket", "hammer", "candle",
        "ladder", "ocean", "basket", "camera", "feather",
        "guitar", "rabbit", "island", "mirror", "rocket",
        "cookie", "helmet", "lantern", "flower", "suitcase",
        "butterfly", "castle", "newspaper", "violin", "backpack",
        "turtle", "snowflake", "diamond", "rainbow", "airplane",
        "anchor", "balloon", "beach", "cactus", "dolphin",
        "eagle", "factory", "glove", "horizon", "jacket",
        "key", "library", "magnet", "notebook", "owl",
        "pillow", "quilt", "rope", "sandal", "teapot",
        "umbrella", "volcano", "whistle", "yacht", "zebra",
        "acorn", "beehive", "compass", "drum", "engine",
        "fountain", "garage", "harvest", "igloo", "jungle",
        "kettle", "lighthouse", "meadow", "nest", "orchard"
    ]
    words = random.sample(WORD_POOL, n_words)

    words_position_map = {
        word: position + 1
        for position, word in enumerate(words)
    }
    return words_position_map

Persona generation (sample of 2)

In [3]:
# df = pd.read_csv("Synthetic_Big5_data.csv")
df = pd.read_csv("../data/persona_checkpoint.csv")
df = df.round(2)
df = df.sample(n=2)

df.insert(0, "gender", df.pop("gender"))
df.rename(columns={"gender": "Gender"}, inplace=True)

# add uniform age - (not perfect since age has a slight impact on Big5, but for this it should be negligible)
df["Age"] = np.random.randint(18, 60, size=len(df))
df.insert(0, "Age", df.pop("Age"))

df["raw_persona"] = df.apply(
    lambda row: "\n".join(f"{col}: {row[col]}" for col in df.columns if col not in ["llm_persona", "persona_generation_date"]),
    axis=1
)

# Assignemnt to split into random groups (for Kahnmemann 1 = AB order, 2 = BA order; and Gamblers Fallacy 1 = red, 2 = black)
rng = np.random.default_rng(42)
df["experiment_group"] = rng.choice(
    [1, 2],
    size=len(df)
)
for n in [20, 40, 80]:
    df[f"serial_position_effect_map_{n}"] = [
        json.dumps(create_word_list(n))
        for _ in range(len(df))
    ]
    
# df.to_csv("2_personas_test.csv", index=False)

In [3]:
# df = df.drop(columns=["llm_persona", "persona_generation_date"])
df = pd.read_csv("generated_data/Personas.csv")
df

,Age,Gender,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,...,Conscientiousness,Neuroticism,Openness_Intellect,raw_persona,experiment_group,serial_position_effect_map_20,serial_position_effect_map_40,serial_position_effect_map_80,llm_persona,persona_generation_date
0,31,male,2.75,3.43,3.64,3.56,2.70,2.94,2.66,2.90,...,2.82,2.78,4.40,Age: 31\nGender: male\nEnthusiasm: 2.75\nAsser...,1,"{""guitar"": 1, ""drum"": 2, ""river"": 3, ""igloo"": ...","{""butterfly"": 1, ""backpack"": 2, ""key"": 3, ""hel...","{""helmet"": 1, ""island"": 2, ""acorn"": 3, ""glove""...",He’s the kind of person who can sit quietly th...,2026-07-27 22:20:58+02:00
1,56,female,3.53,3.04,4.60,3.75,2.69,3.13,3.08,3.71,...,2.91,3.39,3.84,Age: 56\nGender: female\nEnthusiasm: 3.53\nAss...,2,"{""whistle"": 1, ""magnet"": 2, ""teapot"": 3, ""drum...","{""mountain"": 1, ""chair"": 2, ""key"": 3, ""window""...","{""ladder"": 1, ""flower"": 2, ""garage"": 3, ""clock...","Over the years, this woman has built a life th...",2026-07-27 22:20:58+02:00


In [5]:
PERSONA_SYSTEM_PROMPT = get_persona_system_prompt()
runner = LLMExperimentRunner(
    llm="deepseek-v4-pro",
    max_workers=2,
    thinking='enabled',
    reasoning_effort="high",
    )
df_persona = runner.generate_personas(df, PERSONA_SYSTEM_PROMPT)
df_persona

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas: 100%|██████████| 2/2 [01:06<00:00, 33.39s/it]


,Age,Gender,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,...,Conscientiousness,Neuroticism,Openness_Intellect,raw_persona,experiment_group,serial_position_effect_map_20,serial_position_effect_map_40,serial_position_effect_map_80,llm_persona,persona_generation_date
3,31,male,2.75,3.43,3.64,3.56,2.70,2.94,2.66,2.90,...,2.82,2.78,4.40,Age: 31\nGender: male\nEnthusiasm: 2.75\nAsser...,1,"{""guitar"": 1, ""drum"": 2, ""river"": 3, ""igloo"": ...","{""butterfly"": 1, ""backpack"": 2, ""key"": 3, ""hel...","{""helmet"": 1, ""island"": 2, ""acorn"": 3, ""glove""...",He’s the kind of person who can sit quietly th...,2026-07-27 22:20:58+02:00
7,56,female,3.53,3.04,4.60,3.75,2.69,3.13,3.08,3.71,...,2.91,3.39,3.84,Age: 56\nGender: female\nEnthusiasm: 3.53\nAss...,2,"{""whistle"": 1, ""magnet"": 2, ""teapot"": 3, ""drum...","{""mountain"": 1, ""chair"": 2, ""key"": 3, ""window""...","{""ladder"": 1, ""flower"": 2, ""garage"": 3, ""clock...","Over the years, this woman has built a life th...",2026-07-27 22:20:58+02:00


In [10]:
word_count = len(df_persona.iloc[1]["llm_persona"].split())
print(word_count)

667


In [ ]:
# maybe I need something like this in case the model produces more than a single word response
def extract_last_word_clean(
    df: pd.DataFrame,
    columns: list[str],
    suffix: str = "_last_word"
) -> pd.DataFrame:
    for col in columns:
        df[f"{col}{suffix}"] = (
            df[col]
            .str.replace(r"[^\w\s]", "", regex=True)
            .str.strip()
            .str.lower()
            .str.split()
            .str[-1]
        )
    return df

In [ ]:
# old code
# def run_loss_aversion(
#     df,
#     persona,
#     llm,
#     thinking,
#     reasoning_effort,
#     max_workers,
#     use_checkpoint
# ):

#     runner = LLMExperimentRunner(
#         llm=llm,
#         thinking=thinking,
#         reasoning_effort=reasoning_effort,
#         max_workers=max_workers,
#         use_checkpoint=use_checkpoint
#     )
#     results = []
#     for group in [1, 2]:

#         df_group = df[df["experiment_group"] == group].copy()
#         instructions = get_loss_aversion_experiments(
#             persona=persona,
#             group=group
#         )

#         result = runner.run_experiment(
#             df=df_group,
#             instructions=instructions,
#             persona=persona,
#             run_id=f"group_{group}"
#         )
#         results.append(result)
        
#     final_results = pd.concat(results).sort_index()
#     final_results.to_csv(
#         runner.data_dir /
#         f"loss_aversion_{llm}_{persona}_{thinking}.csv",
#         index=False
#     )
#     return None

In [ ]:
# old code for run loss aversion
# Full run for all conditions
# personas = ["description", "raw_persona", "no_persona"]
# thinking_options = ["enabled", "disabled"]
# llms = ["deepseek-v4-pro", "deepseek-v4-flash"]

# for persona in personas:
#     for thinking in thinking_options:
#         for llm in llms:
#             run_loss_aversion(
#                 df=df,
#                 persona=persona,
#                 llm=llm,
#                 thinking=thinking,
#                 reasoning_effort="high",
#                 max_workers=10,
#                 use_checkpoint=True
#             )

In [ ]:
def main_run_experiment(
    df,
    conditions,
    max_workers,
    use_checkpoint
):

    for experiment_name in conditions["experiments"]:
        experiment_results = []
        for persona in conditions["personas"]:
            for thinking in conditions["thinking_options"]:
                for llm in conditions["llms"]:

                    runner = LLMExperimentRunner(
                        llm=llm,
                        thinking=thinking,
                        reasoning_effort=conditions["reasoning_effort"] if thinking == "enabled" else None,
                        max_workers=max_workers,
                        use_checkpoint=use_checkpoint
                    )
                    print(f"Condition: experiment={experiment_name}, persona={persona}, llm={llm}, thinking={thinking}")
                    if experiment_name in ["loss_aversion", "gamblers_fallacy"]:

                        results = []
                        # Group 1 the instructions are in order AB and in Group 2 they are BA (the text of the instructions)
                        # Group 1 the instructions for the previousoutcomes are red and in Group 2 they are black
                        for group in [1,2]:
                            df_group = df[df["experiment_group"] == group].copy()

                            if experiment_name == "loss_aversion":
                                instructions = get_loss_aversion_experiments(
                                    persona=persona,
                                    group=group
                                )
                            else:
                                instructions = get_gamblers_fallacy_experiments(
                                    max_previous_outcomes=conditions["max_previous_outcomes"],
                                    persona=persona,
                                    group=group
                                )

                            result = runner.run_experiment(
                                df=df_group,
                                instructions=instructions,
                                persona=persona,
                                run_id=f"group_{group}"
                            )

                            results.append(result)

                        result = pd.concat(results).sort_index()

                    else:

                        if experiment_name == "serial_position_effect":
                            instructions = get_serial_position_effect_experiments(
                                persona
                            )

                        else:
                            instructions = get_big_five_aspects_questionaire(
                                persona
                            )

                        result = runner.run_experiment(
                            df=df,
                            instructions=instructions,
                            persona=persona,
                            run_id=""
                            )


                    result["llm"] = llm
                    result["persona"] = persona
                    result["thinking"] = thinking
                    experiment_results.append(result)


        final = pd.concat(experiment_results)

        final.to_csv(
            runner.data_dir /
            f"{experiment_name}_results.csv",
            index=False
        )

In [6]:
CONDITIONS = {
    "experiments": ["loss_aversion", "gamblers_fallacy", "serial_position_effect", "big_five_evaluation"],
    "personas": ["description", "raw_persona", "no_persona"],
    "thinking_options": ["enabled", "disabled"],
    "llms": ["deepseek-v4-pro", "deepseek-v4-flash"],
    "max_previous_outcomes": 10,
    "reasoning_effort": "high"

}
main_run_experiment(
    df=df,
    conditions=CONDITIONS,
    max_workers=2,
    use_checkpoint=False
)

Condition: experiment=loss_aversion, persona=description, llm=deepseek-v4-pro, thinking=enabled


Experiment:: 100%|██████████| 16/16 [02:49<00:00, 10.61s/it]


Condition: experiment=loss_aversion, persona=description, llm=deepseek-v4-flash, thinking=enabled


Experiment:: 100%|██████████| 16/16 [01:48<00:00,  6.80s/it]


Condition: experiment=loss_aversion, persona=description, llm=deepseek-v4-pro, thinking=disabled


Experiment:: 100%|██████████| 16/16 [00:18<00:00,  1.13s/it]


... (86 outputs left out)

Condition: experiment=big_five_evaluation, persona=no_persona, llm=deepseek-v4-pro, thinking=disabled


Experiment:: 100%|██████████| 40/40 [00:58<00:00,  1.45s/it]


Condition: experiment=big_five_evaluation, persona=no_persona, llm=deepseek-v4-flash, thinking=disabled


Experiment:: 100%|██████████| 40/40 [00:43<00:00,  1.10s/it]


test this below, runs both groups in parallel:

In [ ]:
def main_run_experiment(
    df,
    conditions,
    max_workers,
    use_checkpoint
):

    for experiment_name in conditions["experiments"]:
        experiment_results = []
        for persona in conditions["personas"]:
            for thinking in conditions["thinking_options"]:
                for llm in conditions["llms"]:
                    workers = max(1, max_workers // 2) if experiment_name in ["loss_aversion", "gamblers_fallacy"] else max_workers
                    runner = LLMExperimentRunner(
                        llm=llm,
                        thinking=thinking,
                        reasoning_effort=conditions["reasoning_effort"] if thinking == "enabled" else None,
                        max_workers=workers,
                        use_checkpoint=use_checkpoint
                    )
                    print(f"Condition: experiment={experiment_name}, persona={persona}, llm={llm}, thinking={thinking}")
                    if experiment_name in ["loss_aversion", "gamblers_fallacy"]:
                        # Group 1 the instructions are in order AB and in Group 2 they are BA (the text of the instructions)
                        # Group 1 the instructions for the previousoutcomes are red and in Group 2 they are black

                        def run_group(group):
                            df_group = df[df["experiment_group"] == group].copy()

                            if experiment_name == "loss_aversion":
                                instructions = get_loss_aversion_experiments(
                                    persona=persona,
                                    group=group
                                )
                            else:
                                instructions = get_gamblers_fallacy_experiments(
                                    max_previous_outcomes=conditions["max_previous_outcomes"],
                                    persona=persona,
                                    group=group
                                )

                            return runner.run_experiment(
                                df=df_group,
                                instructions=instructions,
                                persona=persona,
                                run_id=f"group_{group}"
                            )

                        with ThreadPoolExecutor(max_workers=2) as executor:
                            results = list(executor.map(run_group, [1, 2]))

                        result = pd.concat(results).sort_index()

                    else:

                        if experiment_name == "serial_position_effect":
                            instructions = get_serial_position_effect_experiments(
                                persona
                            )

                        else:
                            instructions = get_big_five_aspects_questionaire(
                                persona
                            )

                        result = runner.run_experiment(
                            df=df,
                            instructions=instructions,
                            persona=persona,
                            run_id=""
                            )


                    result["llm"] = llm
                    result["persona"] = persona
                    result["thinking"] = thinking
                    experiment_results.append(result)


        final = pd.concat(experiment_results)

        final.to_csv(
            runner.data_dir /
            f"{experiment_name}_results.csv",
            index=False
        )

In [ ]:
# #test
# def main_run_experiment(
#     df,
#     conditions,
#     max_workers,
#     use_checkpoint
# ):
#     llm = "deepseek-v4-pro"
#     thinking = "enabled"
#     for experiment_name in conditions["experiments"]:
#         experiment_results = []
#         for persona in conditions["personas"]:


#                     runner = LLMExperimentRunner(
#                         llm=llm,
#                         thinking=thinking,
#                         reasoning_effort=conditions["reasoning_effort"] if thinking == "enabled" else None,
#                         max_workers=max_workers,
#                         use_checkpoint=use_checkpoint
#                     )
#                     print(f"Condtion: {experiment_name}_{persona}_{llm}_thinking_{thinking}")
#                     if experiment_name in ["loss_aversion", "gamblers_fallacy"]:

#                         results = []
#                         # Group 1 the instructions are in order AB and in Group 2 they are BA (the text of the instructions)
#                         # Group 1 the instructions for the previousoutcomes are red and in Group 2 they are black
#                         for group in [1,2]:
#                             df_group = df[df["experiment_group"] == group].copy()

#                             if experiment_name == "loss_aversion":
#                                 instructions = get_loss_aversion_experiments(
#                                     persona=persona,
#                                     group=group
#                                 )
#                             else:
#                                 instructions = get_gamblers_fallacy_experiments(
#                                     max_previous_outcomes=conditions["max_previous_outcomes"],
#                                     persona=persona,
#                                     group=group
#                                 )

#                             result = runner.run_experiment(
#                                 df=df_group,
#                                 instructions=instructions,
#                                 persona=persona,
#                                 run_id=f"group_{group}"
#                             )

#                             results.append(result)

#                         result = pd.concat(results).sort_index()

#                     else:

#                         if experiment_name == "serial_position_effect":
#                             instructions = get_serial_position_effect_experiments(
#                                 persona
#                             )

#                         else:
#                             instructions = get_big_five_aspects_questionaire(
#                                 persona
#                             )

#                         result = runner.run_experiment(
#                             df=df,
#                             instructions=instructions,
#                             persona=persona,
#                             run_id=""
#                             )


#                     result["experiment"] = experiment_name
#                     result["llm"] = llm
#                     result["persona"] = persona
#                     result["thinking"] = thinking
#                     experiment_results.append(result)


#         final = pd.concat(experiment_results)

#         final.to_csv(
#             runner.data_dir /
#             f"{experiment_name}_results.csv",
#             index=False
#         )